In [ ]:
import pandas as pd
import numpy as np

#0. `pandas` and preliminary knowledge

In this exercise we will be using the [`pandas`](https://pandas.pydata.org/docs/) library.

You can use a [cheat sheet of the basic functions](https://pandas.pydata.org/Pandas_Cheat_Sheet.pdf) if you are not familiar with `pandas`.

`pandas` is a popular data manipulation library for Python. It provides data structures like Series and DataFrame alongside the essential functionality required for cleaning, aggregating, transforming, visualizing, and more tasks on data. `pandas` is built on top of the [`numpy`](https://numpy.org/) library, offering a higher-level, more intuitive interface for data analysis and manipulation. It's especially suitable for working with structured data, including datasets from Excel or CSV files, SQL tables, and more. With `pandas`, users can perform tasks like handling missing data, merging and joining datasets, filtering and reshaping data, and many other complex data operations. The extensive functionality of `pandas` makes it an essential tool for data scientists, analysts, and researchers working in Python.


# 1. Load the data
Find a way to load the data into a pandas dataframe. You can find the data here: https://osf.io/fv8c3.

You need to find a way to load this data. There is a function in `pandas` that can help you. If you can't find it on your own or you feel unsure, you can click [here](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html) to find the correct function you can use.

In [ ]:
df = pd.read_csv("https://drive.switch.ch/index.php/s/UEpTFv2Bfa5C1dd/download")
df.head()

# 2. Clean the data
Here we use a very simple approach to clean the data. We remove all the rows that contain missing values. You can try a more sophisticated approach if you want.


In `pandas`, `NaN` (Not a Number) values represent missing or undefined data. `NaN` values can occur when loading datasets, during data manipulation, or when performing calculations. They are often used as a placeholder for missing or unrepresentable data.

Handling `NaN` values is essential as they can affect the results of data analysis and machine learning algorithms. Here are a few ways to handle `NaN` values in pandas:
- You can use the [`dropna()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.dropna.html) method to remove any rows or columns that contain NaN values from a DataFrame.
- You can use the [`fillna()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.fillna.html) method to replace `NaN` values with a specific value or a method (like forward fill or backward fill).

For simplicity's sake we want to just remove all rows that have `NaN` values.

Look up how we could do backfill and forward fill and what doing that would mean in our case. Can you construct a case (on another dataset, or an imagined dataset) where filling is useful?

In [ ]:
df = df.dropna()
df.shape
len(df)

## A Note about `NaN` and `NaT`

Note that `NaN != NaN`!

You can find a detailed explanation of why this is the case [here](https://stackoverflow.com/a/1573715/5320601).

In short:

- `a == b` should hold if `(a - b) == 0`
- but then what is `(a - NaN)`?

The way to check for `NaN` values is to use [`.isna()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.isna.html).

There is also [`NaT`](https://pandas.pydata.org/docs/reference/api/pandas.NaT.html), which is used for missing time values (like a datetime column), and [`NA`](https://pandas.pydata.org/docs/user_guide/missing_data.html#missing-data-na) which is still experimental and whose behaviour could change without warning between versions of `pandas`.

In [ ]:
a = np.nan
b = np.nan

a == b

And important to note, `NaN` is not the same as `None`!

In [ ]:
print(f'{"NaN and None equality:":<30} {np.nan == None}')
print(f'{"None and None equality:":<30} {None == None}')
print(f'{"NaN and NaN equality:":<30} {np.nan == np.nan}')

Keep in mind that this can behave unexpectedly if you are not aware of it.

Here we create a DataFrame with integers, and add a row with a missing value. For the missing value we use `None`.

In [ ]:
_numbers = pd.DataFrame(
    [[0, 1, 2], [3, 4, 5]],
    columns=['a', 'b', 'c']
)
_numbers

We see that this `None` becomes `NaN` in the DataFrame. That is because the dtype of the column is `float64` and `None` is not a number (`NaN`).

In [ ]:
_numbers.loc[2] = [6, 7, None]
_numbers

In [ ]:
_numbers.dtypes

When we do the same with strings though, this is different.

In [ ]:
_strings = pd.DataFrame(
    [['Apple', 'red'], ['Banana', 'yellow']],
    columns=['Fruit', 'Color']
)
_strings

In [ ]:
_strings.loc[2] = ['Cherry', None]
_strings

In [ ]:
_strings.dtypes

`None` can exist in an `object` type column. We don't expect only numbers in this case, so it is not converted.

# 3. Simple statistics
Calculate the mean, median, min and maximum values for all columns.

*Hint*: You can find this in the cheat sheet, in the section 'Summarize Data'.

If you already know how to do this the simple/automatic way, try to calculate these values by hand.

In [ ]:
df.describe()
numeric_columns = df.select_dtypes(include=[np.number]).columns
numeric_columns
df[numeric_columns].head()
df[numeric_columns].median()  # by default column-wise!

# 4. Average cards per game
Calculate the average number of yellow and red cards per game for each player.

Then print out the 5 players with the highest average number of cards per game.

## 4.1 Count the number of cards each player has gotten.
As an intermediate step, let's first calculate the number of cards each player has gotten.

*Hint*: You can access the yellow cards with `df['yellowCards']` and the red cards with `df['redCards']`. How can you calculate the sum of two integers? Can you apply your knowledge to working with DataFrames?

In [ ]:
df['total_cards'] = df['yellowCards'] + df['redCards']
df['total_cards'].head()

## 4.2 Calculate the average number of cards per game for each player.
Next, we can now use this column to calculate the average number of cards per game for each player.

*Hint*: If you don't know how to create new columns yet check the cheat sheet. Save the result of the previous task in a column, and then use that new column as a helper to calculate the average for each player.

In [ ]:
df['avg_cards_per_game'] = df['total_cards'] / df['games']
df['avg_cards_per_game'].head()

## 4.3 Sort the players by the average number of cards per game.
Then we sort by this column.

*Hint*: Check the cheat sheet for how to sort!

In [ ]:
avg_cards_per_game_df = df.sort_values(by='avg_cards_per_game', ascending=False)
avg_cards_per_game_df

## 4.4 Print out the top 5 players.
This is now very easy to do. We will not use `.head()` this time though.

*Hint*: How would you solve this if you were working with a sorted list?

In [ ]:
avg_cards_per_game_df[:5]

# 5. Average number of cards per country
Do the same as in task 4. but this time for each country. This means we need to group the countries!

## 5.1 Group the data by country.

*Hint*: Check the cheat sheet for grouping!

In [ ]:
avg_cards_per_game_df.iloc[:1, 3]

In [ ]:
grouped_by_country = df.groupby('leagueCountry')
grouped_by_country.groups.keys()

## 5.2 Calculate the average number of cards per game for each country.

*Hint*: Remember how you calculated the average number of cards per game for each player! It's not exactly the same here but similar.

In [ ]:
(grouped_by_country['yellowCards'].sum() + grouped_by_country['yellowReds'].sum() + grouped_by_country['redCards'].sum()) / grouped_by_country['games'].sum()

In [ ]:
(grouped_by_country['yellowCards'].sum() + grouped_by_country['redCards'].sum()) / grouped_by_country['games'].sum()

# 6. Correlation
Let's calculate some correlations.

*Hint*: You can use [this](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.corr.html) function if you don't know or can't find out how to do it.

*Hint*: To select numeric columns, you can use `numeric_columns = df.select_dtypes(include=[np.number]).columns`.

In [ ]:
numeric_columns = df.select_dtypes(include=[np.number]).columns
df[numeric_columns].corr #alle Spalten die eine Zahl sind

In [ ]:
correlation = df['height'].corr(df['yellowCards'])
correlation

In [ ]:
df[numeric_columns]

## 6.1 Highest correlations per variable
For each of the variables, let's find the variable that has the highest correlation with it.

*Hint*: Break this down into multiple smaller problems!
- How can you calculate the correlations? (You have done this already!)
- How can you iterate over the columns?
- How can you select each column to get the correlations?
- How can you sort the correlation values?
- How can you find the highest correlation value?

In [ ]:
c = df[numeric_columns].corr().replace(1.0, np.nan) #Correlation mit sich selbst ist immer 1, daher kann man diese eigentlich ?berschreiben
cdf = pd.DataFrame(
    {
        "other": c.idxmax(),
        "correlation":c.max()
    }
)

cdf.sort_values("correlation", ascending=False)


In [ ]:
c.idxmax()

In [ ]:
# df[numeric_columns].corr().replace(1.0, np.nan).sort_values("height", ascending=False)
corrs = df[numeric_columns].corr()

for collumn in numeric_columns:
    high_corrs = corrs[collumn].sort_values(ascending=False)[1:1+3] 
    for index, corr in high_corrs.items():  
        print(collumn, index, corr)

## 6.2 Interesting correlations
Pick out some correlations and explain why you think they are interesting and what might be the cause of them.

# 7. Simple Analysis
Let's do a simple analysis of skin color and the average number of cards. For this we are going to use a boxplot.

Here is an example boxplot. You can adjust the code in the cell below to solve this task. You need to modify the code a bit to use the actual data from the DataFrame.

![](https://www.simplypsychology.org/wp-content/uploads/box-whisker-plot.jpg)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Generate example data
np.random.seed(0)
avg_skin_color = np.random.choice(['Light', 'Medium', 'Dark'], 100)
avg_cards_per_game = np.random.rand(100) * 5  # random values between 0 and 5

# Create a DataFrame
boxplot_values = pd.DataFrame({
    'avg_skin_color': avg_skin_color,
    'avg_cards_per_game': avg_cards_per_game
})

# Grouping and preparing data for boxplot
grouped = boxplot_values.groupby('avg_skin_color')['avg_cards_per_game']

labels = sorted(boxplot_values['avg_skin_color'].unique())

data_for_boxplot = [grouped.get_group(label).values for label in labels]

# Plotting the boxplot
plt.figure(figsize=(10, 6))
plt.boxplot(data_for_boxplot, labels=labels, notch=True)
plt.title('Boxplot of avg_cards_per_game grouped by avg_skin_color')
plt.xlabel('Average Skin Color')
plt.ylabel('Average Cards Per Game')
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Generate example data
np.random.seed(0)

grouped_by_country = df.groupby('leagueCountry')

avg_skin_color = df.groupby('')
avg_cards_per_game = np.random.rand(100) * 5  # random values between 0 and 5

# Create a DataFrame
boxplot_values = pd.DataFrame({
    'avg_skin_color': avg_skin_color,
    'avg_cards_per_game': avg_cards_per_game
})

# Grouping and preparing data for boxplot
grouped = boxplot_values.groupby('avg_skin_color')['avg_cards_per_game']

labels = sorted(boxplot_values['avg_skin_color'].unique())

data_for_boxplot = [grouped.get_group(label).values for label in labels]

# Plotting the boxplot
plt.figure(figsize=(10, 6))
plt.boxplot(data_for_boxplot, labels=labels, notch=True)
plt.title('Boxplot of avg_cards_per_game grouped by avg_skin_color')
plt.xlabel('Average Skin Color')
plt.ylabel('Average Cards Per Game')
plt.show()